# Format Comparison: MCQ vs OSQ

This notebook demonstrates comparing evaluation results between MCQ and OSQ formats.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from metaeval.compare.analysis import FormatComparator, compare_formats
from metaeval.compare.stats.correlation import pearson_correlation, spearman_correlation
from metaeval.compare.stats.effects import cohens_d

## Creating Aligned Data

Format comparison requires aligned MCQ and OSQ results for the same questions.

In [ ]:
np.random.seed(42)

def create_aligned_data(n_questions=100, n_models=3):
    """Create synthetic aligned MCQ/OSQ data."""
    data = []
    
    for model_idx in range(n_models):
        model = f"model_{model_idx}"
        # Different models have different capabilities
        mcq_skill = 0.6 + model_idx * 0.1  # Varies from 0.6 to 0.8
        osq_skill = 0.5 + model_idx * 0.15  # Varies from 0.5 to 0.8
        
        for q_id in range(n_questions):
            # MCQ score (binary)
            mcq_correct = np.random.random() < mcq_skill
            
            # OSQ score (0-100, correlated with MCQ)
            base_osq = osq_skill * 100
            noise = np.random.normal(0, 15)
            # Higher OSQ for correct MCQ answers
            osq_boost = 10 if mcq_correct else -5
            osq_score = np.clip(base_osq + noise + osq_boost, 0, 100)
            
            data.append({
                'question_id': q_id,
                'model': model,
                'mcq_score': float(mcq_correct),
                'osq_score': osq_score,
            })
    
    return pd.DataFrame(data)

aligned_df = create_aligned_data()
print(f"Created aligned dataset: {len(aligned_df)} rows")
aligned_df.head()

## Quick Format Comparison

In [ ]:
# Quick comparison for one model
result = compare_formats(aligned_df, 'model_0')

print(f"Model: model_0")
print(f"MCQ Accuracy: {result['mcq_accuracy']:.3f}")
print(f"OSQ Mean Score: {result['osq_mean']:.1f}/100")
print(f"Correlation: {result['correlation']:.3f}")

## Detailed Analysis with FormatComparator

In [ ]:
# Create comparator
comparator = FormatComparator(aligned_df)

# Analyze single model
report = comparator.analyze('model_0')

print("="*60)
print(f"Format Comparison Report: {report.model}")
print("="*60)

print(f"\nPerformance Metrics:")
print(f"  MCQ Accuracy: {report.mcq_accuracy:.4f}")
print(f"  OSQ Normalized: {report.osq_normalized:.4f}")
print(f"  N Questions: {report.n_questions}")

print(f"\nCorrelations:")
for name, corr in report.correlations.items():
    print(f"  {name}: r={corr['coefficient']:.4f} (p={corr['p_value']:.4f})")

print(f"\nEffect Sizes:")
for name, effect in report.effect_sizes.items():
    print(f"  {name}: {effect['value']:.4f} ({effect['interpretation']})")

## Compare All Models

In [ ]:
# Analyze all models
all_reports = comparator.analyze_all_models()

# Summary table
summary_data = []
for model, report in all_reports.items():
    summary_data.append({
        'Model': model,
        'MCQ Accuracy': f"{report.mcq_accuracy:.3f}",
        'OSQ Normalized': f"{report.osq_normalized:.3f}",
        'Pearson r': f"{report.correlations['pearson']['coefficient']:.3f}",
        'Spearman rho': f"{report.correlations['spearman']['coefficient']:.3f}",
        "Cohen's d": f"{report.effect_sizes['cohens_d']['value']:.3f}",
    })

summary_df = pd.DataFrame(summary_data)
print("Model Comparison Summary:")
summary_df

## Visualizing Format Differences

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: MCQ vs OSQ scores
for model in aligned_df['model'].unique():
    model_data = aligned_df[aligned_df['model'] == model]
    axes[0, 0].scatter(
        model_data['mcq_score'] + np.random.uniform(-0.05, 0.05, len(model_data)),
        model_data['osq_score'],
        alpha=0.3,
        label=model
    )
axes[0, 0].set_xlabel('MCQ Score')
axes[0, 0].set_ylabel('OSQ Score')
axes[0, 0].set_title('MCQ vs OSQ Scores')
axes[0, 0].legend()

# Plot 2: MCQ accuracy by model
models = list(all_reports.keys())
mcq_acc = [all_reports[m].mcq_accuracy for m in models]
osq_norm = [all_reports[m].osq_normalized for m in models]

x = np.arange(len(models))
width = 0.35
axes[0, 1].bar(x - width/2, mcq_acc, width, label='MCQ')
axes[0, 1].bar(x + width/2, osq_norm, width, label='OSQ (normalized)')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_title('MCQ vs OSQ Performance by Model')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(models)
axes[0, 1].legend()

# Plot 3: Correlation coefficients
pearson = [all_reports[m].correlations['pearson']['coefficient'] for m in models]
spearman = [all_reports[m].correlations['spearman']['coefficient'] for m in models]
axes[1, 0].bar(x - width/2, pearson, width, label='Pearson')
axes[1, 0].bar(x + width/2, spearman, width, label='Spearman')
axes[1, 0].set_ylabel('Correlation')
axes[1, 0].set_title('MCQ-OSQ Correlation by Model')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(models)
axes[1, 0].legend()

# Plot 4: OSQ distribution by MCQ correctness
for model in ['model_1']:
    model_data = aligned_df[aligned_df['model'] == model]
    correct = model_data[model_data['mcq_score'] == 1]['osq_score']
    incorrect = model_data[model_data['mcq_score'] == 0]['osq_score']
    
    axes[1, 1].hist(correct, bins=20, alpha=0.5, label='MCQ Correct')
    axes[1, 1].hist(incorrect, bins=20, alpha=0.5, label='MCQ Incorrect')
axes[1, 1].set_xlabel('OSQ Score')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('OSQ Distribution by MCQ Outcome (model_1)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## Statistical Tests

In [ ]:
# Direct use of statistical functions
model_data = aligned_df[aligned_df['model'] == 'model_0']
mcq = model_data['mcq_score'].values
osq = model_data['osq_score'].values

# Pearson correlation
pearson_result = pearson_correlation(mcq, osq)
print(f"Pearson Correlation:")
print(f"  r = {pearson_result.coefficient:.4f}")
print(f"  p-value = {pearson_result.p_value:.6f}")

# Spearman correlation
spearman_result = spearman_correlation(mcq, osq)
print(f"\nSpearman Correlation:")
print(f"  rho = {spearman_result.coefficient:.4f}")
print(f"  p-value = {spearman_result.p_value:.6f}")

In [ ]:
# Cohen's d for paired samples
# Comparing OSQ scores between correct and incorrect MCQ answers
correct_osq = model_data[model_data['mcq_score'] == 1]['osq_score'].values
incorrect_osq = model_data[model_data['mcq_score'] == 0]['osq_score'].values

d = cohens_d(correct_osq, incorrect_osq, paired=False)
print(f"\nCohen's d (correct vs incorrect MCQ):")
print(f"  d = {d.value:.4f}")
print(f"  Interpretation: {d.interpretation}")

## CLI Alternative

```bash
# Run format comparison analysis
metaeval analyze compare results.csv -o analysis/

# Generate markdown report
metaeval analyze compare results.csv --format markdown

# Generate JSON output
metaeval analyze compare results.csv --format json
```